# LayerNorm Analysis

Deep analysis of LayerNorm behavior: gain/bias decomposition, feature amplification/suppression, norm statistics across layers, directional effects, and the LayerNorm Jacobian.

## Why This Matters

Norm layernorm analyzes the scale and magnitude of internal representations. Representation norms carry meaningful signal — they indicate component importance, reveal outlier dimensions, and constrain what information can be transmitted through the residual stream.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.layernorm_analysis import (
    gain_bias_decomposition,
    feature_amplification,
    norm_statistics,
    directional_effects,
    layernorm_jacobian,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

In [ ]:
# Gain/bias decomposition
result = gain_bias_decomposition(model, tokens, layer=0)
print(f'Scale factor (std): {result["scale_factor"]:.4f}')
print(f'Input mean: {result["input_mean"]:.4f}')
print(f'Gain norm: {result["gain_norm"]:.4f}')
print(f'Bias norm: {result["bias_norm"]:.4f}')

In [ ]:
# Feature amplification/suppression
result = feature_amplification(model, tokens, layer=0)
print(f'Mean amplification: {result["mean_amplification"]:.4f}')
print(f'Amplified: {result["n_amplified"]}, Suppressed: {result["n_suppressed"]}')
print('Top amplified:', result['amplified_dims'][:3])
print('Top suppressed:', result['suppressed_dims'][:3])

In [ ]:
# Norm statistics across layers
result = norm_statistics(model, tokens)
print(f'Norm growth trend: {result["norm_growth_trend"]:.4f}')
for layer in result['per_layer']:
    print(f'  Layer {layer["layer"]}: norm={layer["mean_norm"]:.4f}, '
          f'var={layer["mean_variance"]:.4f}')

In [ ]:
# Directional effects
result = directional_effects(model, tokens, layer=0)
print(f'Pre-LN projection: {result["pre_projection"]:.4f}')
print(f'Post-LN projection: {result["post_projection"]:.4f}')
print(f'Projection ratio: {result["projection_ratio"]:.4f}')
print(f'Direction preservation: {result["direction_preservation"]:.4f}')

In [ ]:
# LayerNorm Jacobian
result = layernorm_jacobian(model, tokens, layer=0)
print(f'Jacobian norm: {result["jacobian_norm"]:.4f}')
print(f'Effective rank: {result["effective_rank"]:.2f}')
print(f'Condition number: {result["condition_number"]:.2f}')
print(f'Top singular values: {[f"{s:.4f}" for s in result["top_singular_values"]]}')